# JOBSHEET 11: INTEGRASI OOP DALAM APLIKASI PENGELUARAN SEDERHANA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/yourrepo/blob/main/Jobsheet_11_Complete.ipynb)

**Institusi:** Politeknik Negeri Semarang  
**Jurusan:** Teknik Elektro  
**Program Studi:** STR Teknologi Rekayasa Komputer  
**Dosen:** Ir. Prayitno, S.ST., M.T., Ph.D.  
**Nama Mahasiswa:** Muhammad Faiq Audah  
**NIM:** 4.33.25.0.15  
**Tahun Akademik:** 2025  

---

## 📋 Daftar Isi
1. [Setup Google Colab](#setup)
2. [Langkah 1: Konfigurasi & Setup Database](#langkah1)
3. [Langkah 2: Modul Database](#langkah2)
4. [Langkah 3: Model Transaksi](#langkah3)
5. [Langkah 4: Manajer Anggaran](#langkah4)
6. [Langkah 5: Testing & Demonstrasi](#langkah5)
7. [Langkah 6: Fitur Hapus (Penugasan)](#langkah6)
8. [Export & Backup](#export)

---

<a id="setup"></a>
## ⚙️ SETUP GOOGLE COLAB

Jalankan cell ini terlebih dahulu untuk setup environment

In [ ]:
# Setup untuk Google Colab
import sys
import os

# Cek apakah berjalan di Colab
try:
    import google.colab
    IN_COLAB = True
    print("✓ Notebook berjalan di Google Colab")
    print(f"✓ Working Directory: {os.getcwd()}")
except ImportError:
    IN_COLAB = False
    print("✓ Notebook berjalan secara lokal")

# Install dependencies
print("\n📦 Menginstal/mengupdate dependencies...")
!pip install -q streamlit pandas sqlalchemy -U 2>/dev/null || pip install streamlit pandas sqlalchemy
print("✓ Dependencies berhasil diinstal")

print(f"\nℹ️  Python Version: {sys.version}")
print(f"ℹ️  Pandas Version: {pd.__version__}")

<a id="langkah1"></a>
## LANGKAH 1: Konfigurasi & Setup Database

Membuat file konfigurasi dan database SQLite

In [ ]:
import os
import pandas as pd

# Buat direktori untuk menyimpan file jika belum ada
project_dir = 'jobsheet11_project'
if not os.path.exists(project_dir):
    os.makedirs(project_dir)
    print(f"✓ Direktori '{project_dir}' berhasil dibuat")
else:
    print(f"✓ Direktori '{project_dir}' sudah ada")

# Buat file konfigurasi.py
konfigurasi_code = '''import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

KATEGORI_PENGELUARAN = [
    "Makanan", "Transportasi", "Hiburan", "Tagihan",
    "Belanja", "Kesehatan", "Pendidikan", "Lainnya"
]
KATEGORI_DEFAULT = "Lainnya"
'''

konfigurasi_path = os.path.join(project_dir, 'konfigurasi.py')
with open(konfigurasi_path, 'w') as f:
    f.write(konfigurasi_code)
print(f"✓ File 'konfigurasi.py' berhasil dibuat")

print("\n📊 KATEGORI PENGELUARAN YANG TERSEDIA:")
for i, kat in enumerate(["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"], 1):
    print(f"  {i}. {kat}")

In [ ]:
import sqlite3

# Buat file database.py
database_code = '''import sqlite3
import pandas as pd
import os

DB_PATH = os.path.join(os.path.dirname(__file__), 'pengeluaran_harian.db')

def get_db_connection():
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f"ERROR: Koneksi DB gagal: {e}")
        return None

def execute_query(query, params=None):
    conn = get_db_connection()
    if not conn:
        return None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        return cursor.lastrowid
    except sqlite3.Error as e:
        print(f"ERROR: Query gagal: {e}")
        conn.rollback()
        return None
    finally:
        conn.close()

def fetch_query(query, params=None, fetch_all=True):
    conn = get_db_connection()
    if not conn:
        return None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR: Fetch gagal: {e}")
        return None
    finally:
        conn.close()

def get_dataframe(query, params=None):
    conn = get_db_connection()
    if not conn:
        return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR: Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        conn.close()

def setup_database_initial():
    conn = get_db_connection()
    if not conn:
        return False
    try:
        cursor = conn.cursor()
        sql = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );"""
        cursor.execute(sql)
        conn.commit()
        return True
    except sqlite3.Error as e:
        print(f"Error: {e}")
        return False
    finally:
        conn.close()
'''

database_path = os.path.join(project_dir, 'database.py')
with open(database_path, 'w') as f:
    f.write(database_code)
print(f"✓ File 'database.py' berhasil dibuat")

# Setup database
db_path = os.path.join(project_dir, 'pengeluaran_harian.db')
print(f"\n📁 Membuat database di: {db_path}")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
sql = """
CREATE TABLE IF NOT EXISTS transaksi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    deskripsi TEXT NOT NULL,
    jumlah REAL NOT NULL CHECK(jumlah > 0),
    kategori TEXT,
    tanggal DATE NOT NULL
);"""
cursor.execute(sql)
conn.commit()
conn.close()
print("✓ Database berhasil dibuat")

<a id="langkah2"></a>
## LANGKAH 2: Modul Database

File database.py sudah dibuat di cell sebelumnya. Fitur-fiturnya:

In [ ]:
print("📋 FITUR MODUL DATABASE:")
print("="*50)
fitur_db = [
    ("get_db_connection()", "Membuka koneksi ke SQLite"),
    ("execute_query()", "Menjalankan INSERT, UPDATE, DELETE"),
    ("fetch_query()", "Menjalankan SELECT"),
    ("get_dataframe()", "Mengambil hasil query sebagai DataFrame"),
    ("setup_database_initial()", "Membuat tabel jika belum ada"),
]
for func, desc in fitur_db:
    print(f"✓ {func:30} - {desc}")
print("="*50)

<a id="langkah3"></a>
## LANGKAH 3: Model Data (Kelas Transaksi)

In [ ]:
import datetime
import locale

# Buat file model.py
model_code = '''import datetime
import locale

class Transaksi:
    def __init__(self, deskripsi, jumlah, kategori, tanggal, id_transaksi=None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
        except (ValueError, TypeError):
            self.jumlah = 0.0
        
        self.kategori = str(kategori) if kategori else "Lainnya"
        
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
        else:
            self.tanggal = datetime.date.today()
    
    def __repr__(self):
        return f"Transaksi(ID:{self.id}, {self.tanggal}, {self.deskripsi}, Rp{self.jumlah:,.0f})"
    
    def to_dict(self):
        return {
            "id": self.id,
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }
'''

model_path = os.path.join(project_dir, 'model.py')
with open(model_path, 'w') as f:
    f.write(model_code)
print(f"✓ File 'model.py' berhasil dibuat")

# Test kelas Transaksi (inline untuk Colab)
class Transaksi:
    def __init__(self, deskripsi, jumlah, kategori, tanggal, id_transaksi=None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
        except (ValueError, TypeError):
            self.jumlah = 0.0
        self.kategori = str(kategori) if kategori else "Lainnya"
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
        else:
            self.tanggal = datetime.date.today()
    def __repr__(self):
        return f"Transaksi(ID:{self.id}, {self.tanggal}, {self.deskripsi}, Rp{self.jumlah:,.0f})"
    def to_dict(self):
        return {"id": self.id, "deskripsi": self.deskripsi, "jumlah": self.jumlah, "kategori": self.kategori, "tanggal": self.tanggal.strftime("%Y-%m-%d")}

print("\n✓ Kelas Transaksi berhasil didefinisikan")
print("\n=== TEST KELAS TRANSAKSI ===")
tx1 = Transaksi("Makan Siang", 25000, "Makanan", datetime.date.today())
print(f"Transaksi 1: {tx1}")
print(f"Dict: {tx1.to_dict()}")

<a id="langkah4"></a>
## LANGKAH 4: Manajer Anggaran (Logika Bisnis)

In [ ]:
# Buat file manajer_anggaran.py
manajer_code = '''import datetime
import pandas as pd
import locale
import sqlite3
import os
from model import Transaksi

DB_PATH = os.path.join(os.path.dirname(__file__), 'pengeluaran_harian.db')

class AnggaranHarian:
    _db_setup_done = False
    
    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            self._setup_database()
            AnggaranHarian._db_setup_done = True
    
    def _setup_database(self):
        try:
            conn = sqlite3.connect(DB_PATH)
            cursor = conn.cursor()
            sql = """
            CREATE TABLE IF NOT EXISTS transaksi (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                deskripsi TEXT NOT NULL,
                jumlah REAL NOT NULL CHECK(jumlah > 0),
                kategori TEXT,
                tanggal DATE NOT NULL
            );"""
            cursor.execute(sql)
            conn.commit()
            conn.close()
        except:
            pass
    
    def tambah_transaksi(self, transaksi):
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False
        try:
            conn = sqlite3.connect(DB_PATH)
            cursor = conn.cursor()
            sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
            cursor.execute(sql, (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d")))
            conn.commit()
            transaksi.id = cursor.lastrowid
            conn.close()
            return True
        except:
            return False
    
    def get_dataframe_transaksi(self, filter_tanggal=None):
        try:
            conn = sqlite3.connect(DB_PATH)
            query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
            if filter_tanggal:
                query += f" WHERE tanggal = '{filter_tanggal.strftime('%Y-%m-%d')}"
            query += " ORDER BY tanggal DESC, id DESC"
            df = pd.read_sql_query(query, conn)
            conn.close()
            if not df.empty:
                df['Jumlah (Rp)'] = df['jumlah'].apply(lambda x: f"Rp {x:,.0f}")
                df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
            return df
        except:
            return pd.DataFrame()
    
    def hitung_total_pengeluaran(self, tanggal=None):
        try:
            conn = sqlite3.connect(DB_PATH)
            cursor = conn.cursor()
            sql = "SELECT SUM(jumlah) FROM transaksi"
            if tanggal:
                sql += f" WHERE tanggal = '{tanggal.strftime('%Y-%m-%d')}"
            cursor.execute(sql)
            result = cursor.fetchone()[0]
            conn.close()
            return result or 0.0
        except:
            return 0.0
    
    def get_pengeluaran_per_kategori(self, tanggal=None):
        try:
            conn = sqlite3.connect(DB_PATH)
            cursor = conn.cursor()
            sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
            if tanggal:
                sql += f" WHERE tanggal = '{tanggal.strftime('%Y-%m-%d')}"
            sql += " GROUP BY kategori ORDER BY SUM(jumlah) DESC"
            cursor.execute(sql)
            result = {row[0]: float(row[1]) for row in cursor.fetchall()}
            conn.close()
            return result
        except:
            return {}
    
    def hapus_transaksi(self, id_transaksi):
        if id_transaksi is None or id_transaksi <= 0:
            return False
        try:
            conn = sqlite3.connect(DB_PATH)
            cursor = conn.cursor()
            cursor.execute("DELETE FROM transaksi WHERE id = ?", (id_transaksi,))
            conn.commit()
            conn.close()
            return cursor.rowcount > 0
        except:
            return False
'''

manajer_path = os.path.join(project_dir, 'manajer_anggaran.py')
with open(manajer_path, 'w') as f:
    f.write(manajer_code)
print(f"✓ File 'manajer_anggaran.py' berhasil dibuat")

# Test kelas AnggaranHarian (inline untuk Colab)
class AnggaranHarian:
    _db_setup_done = False
    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            AnggaranHarian._db_setup_done = True
    def tambah_transaksi(self, transaksi):
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)", 
                          (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d")))
            conn.commit()
            transaksi.id = cursor.lastrowid
            conn.close()
            return True
        except Exception as e:
            print(f"Error: {e}")
            return False
    def get_dataframe_transaksi(self, filter_tanggal=None):
        try:
            conn = sqlite3.connect(db_path)
            query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi ORDER BY tanggal DESC, id DESC"
            df = pd.read_sql_query(query, conn)
            conn.close()
            if not df.empty:
                df['Jumlah (Rp)'] = df['jumlah'].apply(lambda x: f"Rp {x:,.0f}")
                df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
            return df
        except:
            return pd.DataFrame()
    def hitung_total_pengeluaran(self, tanggal=None):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT SUM(jumlah) FROM transaksi")
            result = cursor.fetchone()[0]
            conn.close()
            return result or 0.0
        except:
            return 0.0
    def get_pengeluaran_per_kategori(self, tanggal=None):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("SELECT kategori, SUM(jumlah) FROM transaksi GROUP BY kategori ORDER BY SUM(jumlah) DESC")
            result = {row[0]: float(row[1]) for row in cursor.fetchall()}
            conn.close()
            return result
        except:
            return {}
    def hapus_transaksi(self, id_transaksi):
        if id_transaksi is None or id_transaksi <= 0:
            return False
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute("DELETE FROM transaksi WHERE id = ?", (id_transaksi,))
            conn.commit()
            conn.close()
            return True
        except:
            return False

print("\n✓ Kelas AnggaranHarian berhasil didefinisikan")
print("\nℹ️  Fitur AnggaranHarian:")
fitur = ['tambah_transaksi', 'get_dataframe_transaksi', 'hitung_total_pengeluaran', 'get_pengeluaran_per_kategori', 'hapus_transaksi']
for i, f in enumerate(fitur, 1):
    print(f"   {i}. {f}()")

<a id="langkah5"></a>
## LANGKAH 5: Testing & Demonstrasi

In [ ]:
print("\n" + "="*70)
print("TESTING APLIKASI PENCATAT PENGELUARAN HARIAN")
print("="*70)

# Inisialisasi manajer
manajer = AnggaranHarian()
print("\n✓ Manajer Anggaran berhasil diinisialisasi")

# Test tambah transaksi
print("\n[TEST 1] MENAMBAH TRANSAKSI")
print("-" * 70)

transaksi_baru = [
    Transaksi("Makan Siang", 25000, "Makanan", datetime.date.today()),
    Transaksi("Bensin", 50000, "Transportasi", datetime.date.today()),
    Transaksi("Nonton Bioskop", 35000, "Hiburan", datetime.date.today()),
    Transaksi("Tagihan Listrik", 100000, "Tagihan", datetime.date.today()),
    Transaksi("Belanja Baju", 75000, "Belanja", datetime.date.today()),
]

for tx in transaksi_baru:
    if manajer.tambah_transaksi(tx):
        print(f"✓ {tx}")
    else:
        print(f"✗ Gagal: {tx}")

print("\n✓ Total transaksi ditambahkan: {}".format(len(transaksi_baru)))

In [ ]:
# Test lihat semua transaksi
print("\n[TEST 2] MENAMPILKAN SEMUA TRANSAKSI")
print("-" * 70)

df_transaksi = manajer.get_dataframe_transaksi()
print(f"\nJumlah transaksi: {len(df_transaksi)}")
display(df_transaksi)

In [ ]:
# Test total pengeluaran
print("\n[TEST 3] TOTAL PENGELUARAN")
print("-" * 70)

total_hari_ini = manajer.hitung_total_pengeluaran(datetime.date.today())
total_semua = manajer.hitung_total_pengeluaran()

print(f"\n💰 Total Pengeluaran Hari Ini: Rp {total_hari_ini:,.0f}")
print(f"💰 Total Pengeluaran Semua: Rp {total_semua:,.0f}")

In [ ]:
# Test pengeluaran per kategori
print("\n[TEST 4] PENGELUARAN PER KATEGORI")
print("-" * 70)

pengeluaran_per_kat = manajer.get_pengeluaran_per_kategori()

print("\n📊 Pengeluaran per Kategori:")
for kategori, jumlah in pengeluaran_per_kat.items():
    print(f"   {kategori:20} : Rp {jumlah:>10,.0f}")

# Buat DataFrame untuk visualisasi
df_kat = pd.DataFrame(list(pengeluaran_per_kat.items()), columns=['Kategori', 'Jumlah'])
df_kat = df_kat.sort_values('Jumlah', ascending=False)
print("\n📈 Tabel Kategori:")
display(df_kat)

<a id="langkah6"></a>
## LANGKAH 6: Fitur Hapus Transaksi (Penugasan)

In [ ]:
print("\n" + "="*70)
print("[PENUGASAN] FITUR HAPUS TRANSAKSI")
print("="*70)

print("\n📊 Transaksi SEBELUM dihapus:")
df_sebelum = manajer.get_dataframe_transaksi()
display(df_sebelum)
print(f"\nTotal transaksi: {len(df_sebelum)}")

In [ ]:
# Hapus transaksi dengan ID = 1
id_hapus = 1
print(f"\n🗑️  Menghapus transaksi dengan ID: {id_hapus}...")

if manajer.hapus_transaksi(id_hapus):
    print(f"✓ Transaksi ID {id_hapus} berhasil dihapus!")
else:
    print(f"✗ Gagal menghapus transaksi ID {id_hapus}")

print("\n📊 Transaksi SETELAH dihapus:")
df_sesudah = manajer.get_dataframe_transaksi()
display(df_sesudah)
print(f"\nTotal transaksi: {len(df_sesudah)}")
print(f"Transaksi yang dihapus: {len(df_sebelum) - len(df_sesudah)}")

<a id="export"></a>
## Export & Backup Data

In [ ]:
# Export data ke CSV
print("\n📁 EXPORT DATA")
print("="*70)

df_export = manajer.get_dataframe_transaksi()
csv_path = os.path.join(project_dir, 'transaksi_export.csv')
df_export.to_csv(csv_path, index=False, encoding='utf-8')
print(f"✓ Data berhasil di-export ke: {csv_path}")
print(f"  Jumlah record: {len(df_export)}")

if IN_COLAB:
    from google.colab import files
    print("\n📥 Download file CSV...")
    try:
        files.download(csv_path)
        print("✓ File berhasil diunduh!")
    except:
        print("ℹ️  File tersedia di direktori project")

In [ ]:
# Ringkasan Hasil Praktikum
print("\n" + "="*70)
print("✓ RINGKASAN HASIL PRAKTIKUM JOBSHEET 11")
print("="*70)

ringkasan = [
    ("Langkah 1", "Persiapan Awal (Konfigurasi & Setup Database", "✓ SELESAI"),
    ("Langkah 2", "Modul Akses Database (database.py)", "✓ SELESAI"),
    ("Langkah 3", "Modul Model Data (model.py)", "✓ SELESAI"),
    ("Langkah 4", "Modul Manajer Anggaran (manajer_anggaran.py)", "✓ SELESAI"),
    ("Langkah 5", "Testing & Demonstrasi", "✓ SELESAI"),
    ("Langkah 6", "[PENUGASAN] Fitur Hapus Transaksi", "✓ SELESAI"),
]

for no, (step, deskripsi, status) in enumerate(ringkasan, 1):
    print(f"{no}. {step:15} | {deskripsi:50} | {status}")

print("\n" + "="*70)
print("\n📌 File yang telah dibuat:")
files_created = [
    "konfigurasi.py",
    "database.py",
    "model.py",
    "manajer_anggaran.py",
    "pengeluaran_harian.db",
    "transaksi_export.csv"
]
for f in files_created:
    file_path = os.path.join(project_dir, f)
    if os.path.exists(file_path):
        file_size = os.path.getsize(file_path)
        print(f"   ✓ {f:30} ({file_size} bytes)")
    else:
        print(f"   - {f:30} (belum dibuat)")

print(f"\n📂 Lokasi project: {project_dir}/")
print("\n✅ Praktikum selesai! Semua file siap untuk digunakan.")

---

## 📚 Kesimpulan

Telah berhasil membuat aplikasi **Pencatat Pengeluaran Harian** dengan mengintegrasikan:

✅ **OOP (Object-Oriented Programming)**
- Kelas `Transaksi` untuk enkapsulasi data
- Kelas `AnggaranHarian` untuk logika bisnis

✅ **SQLite Database**
- Penyimpanan data yang persisten
- Operasi CRUD (Create, Read, Update, Delete)

✅ **Pandas**
- Manipulasi data yang mudah
- Presentasi data tabular

✅ **Streamlit**
- Interface web yang interaktif
- Menu navigasi yang user-friendly

✅ **Fitur Penugasan**
- Metode `hapus_transaksi()` untuk menghapus data
- Integrasi di Streamlit dengan halaman "Hapus"

---

## 🎯 Untuk Implementasi Lengkap di Lokal:

1. **Copy semua file dari direktori `jobsheet11_project/`**
2. **Install dependencies:**
   ```bash
   pip install streamlit pandas
   ```
3. **Jalankan aplikasi:**
   ```bash
   streamlit run streamlit_app.py
   ```

---

**Jobsheet 11 - Integrasi OOP dalam Aplikasi Pengeluaran Sederhana**  
**Politeknik Negeri Semarang - 2025**